In [5]:
import random

def split_trivial(s, n, k):
    """Faza podziału sekretu metodą trywialną."""
    if s < 0 or s >= k:
        raise ValueError("Sekret musi być z zakresu <0, k-1>")

    shares = [random.randint(0, k-1) for _ in range(n-1)]
 
    sn = (s - sum(shares)) % k
    shares.append(sn)
    
    return shares

def reconstruct_trivial(shares, k):
    """Faza odtwarzania sekretu metodą trywialną."""
    return sum(shares) % k

k_param = 100
sekret_s = 42
liczba_udzialow_n = 5

print(f"Oryginalny sekret: {sekret_s}, k: {k_param}")
udzialy = split_trivial(sekret_s, liczba_udzialow_n, k_param)
print(f"Wygenerowane udziały: {udzialy}")

odtworzony_sekret = reconstruct_trivial(udzialy, k_param)
print(f"Odtworzony sekret: {odtworzony_sekret}")

Oryginalny sekret: 42, k: 100
Wygenerowane udziały: [73, 85, 53, 93, 38]
Odtworzony sekret: 42


**1. Dla jakich wartości metoda ta nie jest bezpieczna?**
Metoda przestaje być bezpieczna, gdy wartość $k$ jest zbyt mała. Jeśli $k$ jest małe, potencjalny atakujący mający $n-1$ udziałów może łatwo przeprowadzić atak siłowy (brute-force), sprawdzając wszystkie możliwe wartości ostatniego udziału od $0$ do $k-1$.

**2. Podstawowe wady trywialnego podziału sekretu:**
* Brak odporności na utratę danych (Brak tolerancji błędów): Ponieważ $t = n$, zgubienie lub zniszczenie choćby jednego udziału sprawia, że odzyskanie sekretu jest matematycznie niemożliwe.
* Problem z elastycznością: Nie można dostosować progu bezpieczeństwa (np. ustalenia, że wystarczą 3 z 5 osób). Muszą być obecni wszyscy uczestnicy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def generate_polynomial(s, t, p):
    """Generuje współczynniki wielomianu: s + a_1*x + ... + a_{t-1}*x^{t-1}"""
   
    coeffs = [s] + [random.randint(1, p-1) for _ in range(t-1)]
    return coeffs

def evaluate_polynomial(coeffs, x, p):
    """Oblicza wartość wielomianu dla danego x modulo p."""
    result = 0
    for i, coeff in enumerate(coeffs):
        result = (result + coeff * (x ** i)) % p
    return result

def split_shamir(s, n, t, p):
    """Dzieli sekret schematem Shamira i zwraca udziały oraz współczynniki."""
    coeffs = generate_polynomial(s, t, p)
    shares = [(x, evaluate_polynomial(coeffs, x, p)) for x in range(1, n+1)]
    return shares, coeffs

def reconstruct_shamir(shares, p):
    """Odtwarza sekret na podstawie t udziałów używając interpolacji Lagrange'a."""
    s_reconstructed = 0
    t = len(shares)
    
    for i in range(t):
        xi, yi = shares[i]
        li = 1
        
        for j in range(t):
            if i != j:
                xj, _ = shares[j]
                numerator = (0 - xj) % p
                denominator = (xi - xj) % p
                
                inv_denominator = pow(denominator, -1, p)
                
                li = (li * numerator * inv_denominator) % p
                
        s_reconstructed = (s_reconstructed + yi * li) % p
        
    return s_reconstructed


In [ ]:

n = 6       # a. całkowita liczba udziałów
t = 3       # b. wymagana liczba udziałów 
s = 42      # c. sekret (wyraz wolny a_0)
p = 101     # d. liczba pierwsza (p > s oraz p > n)

In [8]:

shares, coeffs = split_shamir(s, n, t, p)
print(f"Wygenerowane współczynniki wielomianu: {coeffs}")
print("Rozdane udziały (x, y):")
for share in shares:
    print(share)

# Symulujemy zebranie tylko 't' losowych udziałów 
zebrane_udzialy = random.sample(shares, t)
print(f"Zebrane udziały do rekonstrukcji: {zebrane_udzialy}")

odtworzony = reconstruct_shamir(zebrane_udzialy, p)
print(f"\nOdtworzony sekret: {odtworzony} (Oryginał: {s})")



Wygenerowane współczynniki wielomianu: [42, 21, 30]
Rozdane udziały (x, y):
(1, 93)
(2, 2)
(3, 72)
(4, 0)
(5, 89)
(6, 36)
Zebrane udziały do rekonstrukcji: [(5, 89), (1, 93), (4, 0)]

Odtworzony sekret: 42 (Oryginał: 42)


**Jaka jest minimalna, wymagana liczba udziałów, aby algorytm działał poprawnie?**
Minimalna wymagana liczba udziałów (próg $t$) wynosi 2. 
Gdybyśmy ustawili $t = 1$, wielomian interpolacyjny byłby stopnia zerowego . Byłaby to funkcja stała o wartości naszego sekretu. 